In [2]:
import matplotlib.pyplot as plt
import networkx as nx

class Splitwise:
    def _init_(self, members):
        self.members = members
        self.expenses = []
        self.balances = {member: 0 for member in members}

    def show_balances(self):
        print("\nCurrent Balances:")
        for person, balance in self.balances.items():
            if balance > 0:
                print(f"{person} should receive {balance:.2f}")
            elif balance < 0:
                print(f"{person} owes {-balance:.2f}")
            else:
                print(f"{person} is settled up")
        print("\nExpenses so far:")
        for expense in self.expenses:
          print(expense,"\n")

    def draw_graph(self, edges, title):
        graph = nx.DiGraph()
        graph.add_edges_from(edges)
        pos = nx.spring_layout(graph)
        plt.figure(figsize=(8, 6))
        nx.draw(graph, pos, with_labels=True, node_color='lightblue', node_size=2000, font_size=10, font_weight='bold', edge_color='gray')
        edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in graph.edges(data=True)}
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels)
        plt.title(title)
        plt.show()
    def add_expense(self):
        try:
            payer = input("Enter the name of the payer: ")
            if payer not in self.members:
                print("Payer is not a member of the group.")
                return

            amount = float(input("Enter the amount of the bill: "))
            participants = input("Enter the names of participants (comma-separated): ").split(',')

            participants = [participant.strip() for participant in participants]
            for participant in participants:
                if participant not in self.members:
                    print(f"{participant} is not a member of the group.")
                    return

            share = amount / len(participants)

            if payer not in participants:
              self.balances[payer]+=amount
            for person in participants:
                if person == payer:
                    self.balances[payer] += amount-share
                else:
                    self.balances[person] -= share

            # Store the expense
            self.expenses.append({'amount': amount, 'payer': payer, 'participants': participants})
            print("Expense added successfully!")

        except ValueError:
            print("Invalid input. Please enter a valid amount.")
    def simplify_debts(self):
        settled_balances = {k: v for k, v in self.balances.items() if v != 0}

        creditors = [(k, v) for k, v in settled_balances.items() if v > 0]
        debtors = [(k, -v) for k, v in settled_balances.items() if v < 0]

        print(f"Creditors: {creditors}")
        print(f"Debtors: {debtors}")

        if not creditors:
            print("No creditors available to simplify debts.")
            return

        print("\nSimplified Debts:")
        edges_after = []
        while creditors and debtors:
            creditor, credit_amount = creditors.pop(0)
            debtor, debt_amount = debtors.pop(0)
            settlement = min(credit_amount, debt_amount)
            print(f"{debtor} pays {settlement:.2f} to {creditor}")
            edges_after.append((debtor, creditor, {'weight': settlement}))

            credit_amount -= settlement
            debt_amount -= settlement

            if credit_amount > 0:
                creditors.insert(0, (creditor, credit_amount))
            if debt_amount > 0:
                debtors.insert(0, (debtor, debt_amount))

        print("\nGraph After Simplification:")
        self.draw_graph(edges_after, "Graph After Simplification")
    def settle_up(self):
        debtor = input("Enter the name of the debtor: ")
        creditor = input("Enter the name of the creditor: ")

        try:
            amount = float(input("Enter the amount to settle up: "))

            if debtor not in self.members or creditor not in self.members:
                print("Invalid member(s). Please enter valid group members.")
                return

            if self.balances[debtor] + amount > 0 or self.balances[creditor] - amount < 0:
                print("Settlement amount exceeds the balance. Please enter a valid amount.")
                return

            self.balances[debtor] += amount
            self.balances[creditor] -= amount
            print(f"{debtor} settled {amount:.2f} with {creditor}.")

        except ValueError:
            print("Invalid input. Please enter a valid amount.")

def main():
    print("Welcome to the Splitwise Application!")
    n = int(input("Enter the number of members in the group: "))
    members = [input(f"Enter the name of member {i + 1}: ").strip() for i in range(n)]

    app = Splitwise(members)

    while True:
        print("\nMenu:")
        print("1. Add Expense")
        print("2. Show Balances")
        print("3. Simplify Debts")
        print("4. Settle Up")
        print("5. Exit")

        choice = input("Enter your choice (1-5): ")

        if choice == '1':
            app.add_expense()
        elif choice == '2':
            app.show_balances()
        elif choice == '3':
            app.simplify_debts()
        elif choice == '4':
            app.settle_up()
        elif choice == '5':
            print("Exiting the application. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 5.")

if __name__ == "__main__":
    main()

Welcome to the Splitwise Application!


KeyboardInterrupt: Interrupted by user